# Data Ingestion pipeline

In [ ]:
# imports
import os
from pathlib import Path

import holidays
import logging
import numpy as np
import pandas as pd
import requests

from dotenv import load_dotenv, find_dotenv
import pymongo
from pymongo import MongoClient

In [38]:
# Logging
LOGS_DIR = Path("logs")
LOGS_DIR.mkdir(exist_ok=True)

log = logging.getLogger("pipeline")
log.setLevel(logging.INFO)
log.handlers.clear()
log.propagate = False  
formatter = logging.Formatter("%(asctime)s  %(levelname)-8s  %(message)s", datefmt="%H:%M:%S")

console = logging.StreamHandler()
console.setFormatter(formatter)

file = logging.FileHandler(LOGS_DIR / "pipeline.log", mode="w")
file.setFormatter(formatter)

log.addHandler(console)
log.addHandler(file)

log.info("Logging initialized — file: %s", LOGS_DIR / "pipeline.log")

16:38:14  INFO      Logging initialized — file: logs/pipeline.log


## 2. Configuration

Set up directory paths, file paths, and load the MongoDB connection URI from the `.env` file.

In [39]:
# Config
RAW       = Path("data/raw")
PROCESSED = Path("data/processed")
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

CALLS_RAW    = RAW / "911_calls.csv"
ENRICHED_OUT = PROCESSED / "enriched_calls.csv"


load_dotenv(find_dotenv(), override=True)
MONGO_URI       = os.getenv("MONGO_URI")
DB_NAME         = "austin_911"
COLLECTION_NAME = "calls"

log.info("Raw path         : %s", RAW)
log.info("Processed path   : %s", PROCESSED)
log.info("Calls file exists: %s", CALLS_RAW.exists())
log.info("MongoDB target   : %s > %s", DB_NAME, COLLECTION_NAME)

16:38:17  INFO      Raw path         : data/raw
16:38:17  INFO      Processed path   : data/processed
16:38:17  INFO      Calls file exists: True
16:38:17  INFO      MongoDB target   : austin_911 > calls


In [40]:
# Load raw csv
try:
    df_calls = pd.read_csv(CALLS_RAW)
except FileNotFoundError:
    log.error("911 calls CSV not found at %s", CALLS_RAW)
    raise

log.info("Raw dataset shape: %s", df_calls.shape)
log.info("Columns: %s", list(df_calls.columns))

df_calls.head()

16:58:38  INFO      Raw dataset shape: (3974470, 27)
16:58:38  INFO      Columns: ['Incident Number', 'Incident Type', 'Council District', 'Mental Health Flag', 'Priority Level', 'Response Datetime', 'Response Year', 'Response Month', 'Response Day of Week', 'Response Hour', 'First Unit Arrived Datetime', 'Call Closed Datetime', 'Sector', 'Initial Problem Description', 'Initial Problem Category', 'Final Problem Description', 'Final Problem Category', 'Number of Units Arrived', 'Unit Time on Scene', 'Call Disposition Description', 'Report Written Flag', 'Response Time', 'Officer Injured/Killed Count', 'Subject Injured/Killed Count', 'Other Injured/Killed Count', 'Geo ID', 'Census Block Group']


,Incident Number,Incident Type,Council District,Mental Health Flag,Priority Level,Response Datetime,Response Year,Response Month,Response Day of Week,Response Hour,...,Number of Units Arrived,Unit Time on Scene,Call Disposition Description,Report Written Flag,Response Time,Officer Injured/Killed Count,Subject Injured/Killed Count,Other Injured/Killed Count,Geo ID,Census Block Group
0,260530390,Dispatched Incident,1,Not Mental Health Incident,Priority 2,02/22/2026 06:26:02 AM,2026,Feb,Sun,6,...,2.0,9320.0,No Report,0,905.0,0,0,0,4.845304e+11,4.530440e+09
1,251700545,Dispatched Incident,10,Not Mental Health Incident,Priority 2,06/19/2025 11:25:05 AM,2025,Jun,Thu,11,...,1.0,1490.0,No Report,0,310.0,0,0,0,4.845303e+11,4.530306e+09
2,250401212,Dispatched Incident,2,Not Mental Health Incident,Priority 2,02/09/2025 08:56:39 PM,2025,Feb,Sun,20,...,2.0,2462.0,No Report,0,161.0,0,0,0,4.845398e+11,4.539800e+09
3,182151197,Dispatched Incident,7,Not Mental Health Incident,Priority 2,08/03/2018 03:24:53 PM,2018,Aug,Fri,15,...,2.0,5547.0,No Report,0,702.0,0,0,0,4.845304e+11,4.530440e+09
4,183160413,Officer-Initiated Incident,2,Not Mental Health Incident,Priority 3,11/12/2018 09:04:27 AM,2018,Nov,Mon,9,...,1.0,8.0,No Report,0,NaN,0,0,0,4.845398e+11,4.539800e+09


In [43]:
# Normalize column names: 
df_calls.columns = (
    df_calls.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("/", "_")
)
log.info("Cleaned columns: %s", list(df_calls.columns))

16:59:13  INFO      Cleaned columns: ['incident_number', 'incident_type', 'council_district', 'mental_health_flag', 'priority_level', 'response_datetime', 'response_year', 'response_month', 'response_day_of_week', 'response_hour', 'first_unit_arrived_datetime', 'call_closed_datetime', 'sector', 'initial_problem_description', 'initial_problem_category', 'final_problem_description', 'final_problem_category', 'number_of_units_arrived', 'unit_time_on_scene', 'call_disposition_description', 'report_written_flag', 'response_time', 'officer_injured_killed_count', 'subject_injured_killed_count', 'other_injured_killed_count', 'geo_id', 'census_block_group']


In [44]:
# Keep only the two most urgent priority levels (Priority 0 = most critical)
df_urgent = df_calls[df_calls["priority_level"].isin(["Priority 0", "Priority 1"])].copy()

log.info("Urgent calls shape: %s", df_urgent.shape)
log.info("Remaining missing in final_problem_category: %d",
         (df_urgent["final_problem_category"] == "missing").sum())
log.info("Priority breakdown:\n%s", df_urgent["priority_level"].value_counts())
log.info("Top categories:\n%s", df_urgent["final_problem_category"].value_counts().head(10))

16:59:27  INFO      Urgent calls shape: (858648, 27)
16:59:27  INFO      Remaining missing in final_problem_category: 171
16:59:27  INFO      Priority breakdown:
priority_level
Priority 1    522019
Priority 0    336629
Name: count, dtype: int64
16:59:27  INFO      Top categories:
final_problem_category
Welfare Check          172620
Disturbance            149584
Assistance              83215
Traffic Stop/Hazard     76901
Crashes                 60638
Other                   58276
Simple Assault          45830
Shoot/Stab              41103
Alarms                  38282
Burglary                16880
Name: count, dtype: int64


In [45]:
# Convert the string timestamp to datetime 
try:
    df_urgent["response_datetime"] = pd.to_datetime(
        df_urgent["response_datetime"], format="%m/%d/%Y %I:%M:%S %p"
    )
except ValueError as e:
    log.error("Datetime parsing failed — check the format string: %s", e)
    raise

log.info("response_datetime dtype after parsing: %s", df_urgent["response_datetime"].dtype)

16:59:43  INFO      response_datetime dtype after parsing: datetime64[us]


In [46]:
# Filter to baker district
df_baker = df_urgent[df_urgent["sector"] == "Baker"].copy()

# date column used for joining 
df_baker["date"] = df_baker["response_datetime"].dt.normalize()

# Split train test by calender year 
df_train = df_baker[df_baker["response_datetime"].dt.year == 2025].copy()
df_test  = df_baker[df_baker["response_datetime"].dt.year == 2024].copy()

log.info("Baker total shape  : %s", df_baker.shape)
log.info("Train (2025) shape : %s", df_train.shape)
log.info("Test  (2024) shape : %s", df_test.shape)

17:00:31  INFO      Baker total shape  : (82290, 28)
17:00:31  INFO      Train (2025) shape : (8453, 28)
17:00:31  INFO      Test  (2024) shape : (8946, 28)


In [47]:
# event calenders 

# UT football games
ut_home_games = pd.to_datetime([
    # 2024 season
    "2024-08-31", "2024-09-14", "2024-09-21", "2024-09-28",
    "2024-10-19", "2024-11-09", "2024-11-23",
    # 2025 season
    "2025-09-06", "2025-09-13", "2025-09-27",
    "2025-11-01", "2025-11-22", "2025-11-29",
])

# SXSW
sxsw_dates = pd.to_datetime(
    pd.date_range("2024-03-08", "2024-03-16").tolist() +
    pd.date_range("2025-03-07", "2025-03-15").tolist()
)

# ACL Music Festival
acl_dates = pd.to_datetime([
    "2024-10-04", "2024-10-05", "2024-10-06",
    "2024-10-11", "2024-10-12", "2024-10-13",
    "2025-10-03", "2025-10-04", "2025-10-05",
    "2025-10-10", "2025-10-11", "2025-10-12",
])

# UT Move in
movein_dates = pd.to_datetime([
    "2024-08-23", "2024-08-24", "2024-08-25",
    "2025-08-22", "2025-08-23", "2025-08-24",
])

# UT spring break
spring_break_dates = pd.to_datetime(
    pd.date_range("2024-03-11", "2024-03-16").tolist() +
    pd.date_range("2025-03-17", "2025-03-22").tolist()
)

# Ut graduation
graduation_dates = pd.to_datetime([
    "2024-05-10", "2024-05-11",
    "2025-05-09", "2025-05-10",
])

In [48]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """Attach calendar and Austin-event indicator columns to a per-call dataframe."""
    df = df.copy()

    # Standard calendar features derived from the normalized date
    df["day_of_week"] = df["date"].dt.dayofweek        # 0 = Monday, 6 = Sunday
    df["month"]       = df["date"].dt.month
    df["day_of_year"] = df["date"].dt.dayofyear
    df["is_weekend"]  = df["day_of_week"].isin([5, 6]).astype(int)
    df["season"]      = df["month"].map({
        12: "winter", 1: "winter",  2: "winter",
         3: "spring", 4: "spring",  5: "spring",
         6: "summer", 7: "summer",  8: "summer",
         9: "fall",  10: "fall",   11: "fall",
    })

    # Holidays
    us_holidays = holidays.US(state="TX", years=df["date"].dt.year.unique())
    df["is_holiday"] = df["date"].dt.date.isin(us_holidays).astype(int)

    # Event specific flags
    df["is_ut_game"]      = df["date"].isin(ut_home_games).astype(int)
    df["is_sxsw"]         = df["date"].isin(sxsw_dates).astype(int)
    df["is_acl"]          = df["date"].isin(acl_dates).astype(int)
    df["is_ut_movein"]    = df["date"].isin(movein_dates).astype(int)
    df["is_spring_break"] = df["date"].isin(spring_break_dates).astype(int)
    df["is_graduation"]   = df["date"].isin(graduation_dates).astype(int)

    return df

In [49]:
# Apply features
df_train = add_features(df_train)
df_test  = add_features(df_test)

for name, df in [("TRAIN", df_train), ("TEST", df_test)]:
    log.info("── %s ──────────────────────────────", name)
    log.info("  Total calls               : %d", len(df))
    log.info("  Calls on holidays         : %d", df["is_holiday"].sum())
    log.info("  Calls on UT game days     : %d", df["is_ut_game"].sum())
    log.info("  Calls during SXSW        : %d", df["is_sxsw"].sum())
    log.info("  Calls during ACL          : %d", df["is_acl"].sum())
    log.info("  Calls on move-in days     : %d", df["is_ut_movein"].sum())
    log.info("  Calls during spring break : %d", df["is_spring_break"].sum())
    log.info("  Calls on graduation days  : %d", df["is_graduation"].sum())

17:04:22  INFO      ── TRAIN ──────────────────────────────
17:04:22  INFO        Total calls               : 8453
17:04:22  INFO        Calls on holidays         : 370
17:04:22  INFO        Calls on UT game days     : 154
17:04:22  INFO        Calls during SXSW        : 245
17:04:22  INFO        Calls during ACL          : 146
17:04:22  INFO        Calls on move-in days     : 80
17:04:22  INFO        Calls during spring break : 130
17:04:22  INFO        Calls on graduation days  : 57
17:04:22  INFO      ── TEST ──────────────────────────────
17:04:22  INFO        Total calls               : 8946
17:04:22  INFO        Calls on holidays         : 406
17:04:22  INFO        Calls on UT game days     : 223
17:04:22  INFO        Calls during SXSW        : 207
17:04:22  INFO        Calls during ACL          : 135
17:04:22  INFO        Calls on move-in days     : 80
17:04:22  INFO        Calls during spring break : 142
17:04:22  INFO        Calls on graduation days  : 41


In [50]:
def get_austin_weather(start_date: str, end_date: str) -> pd.DataFrame:
    """
    Fetch daily weather for Austin from the Open-Meteo historical archive API.
    Returns one row per day with temperature, precipitation, wind speed, and WMO weather code.
    """
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": 30.2672,
        "longitude": -97.7431,
        "start_date": start_date,
        "end_date": end_date,
        "daily": [
            "temperature_2m_max", "temperature_2m_min",
            "precipitation_sum", "windspeed_10m_max", "weathercode",
        ],
        "timezone": "America/Chicago",
        "temperature_unit": "fahrenheit",
        "windspeed_unit": "mph",
        "precipitation_unit": "inch",
    }

    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status() # raise for http errors
        data = response.json()
    except requests.exceptions.Timeout:
        log.error("Weather API timed out for range %s – %s", start_date, end_date)
        raise
    except requests.exceptions.RequestException as e:
        log.error("Weather API request failed: %s", e)
        raise

    # Convert into df
    df = pd.DataFrame(data["daily"])
    df["date"] = pd.to_datetime(df["time"])
    df = df.drop(columns=["time"])
    return df

In [51]:
# Fetch historical weather for both yrs
try:
    weather_2024 = get_austin_weather("2024-01-01", "2024-12-31")
    weather_2025 = get_austin_weather("2025-01-01", "2025-12-31")
except Exception as e:
    log.error("Failed to fetch weather data: %s", e)
    raise

weather = pd.concat([weather_2024, weather_2025], ignore_index=True)

log.info("Weather shape : %s", weather.shape)
log.info("Weather nulls :\n%s", weather.isnull().sum())
log.info("Sample:\n%s", weather.head())

17:05:24  INFO      Weather shape : (731, 6)
17:05:24  INFO      Weather nulls :
temperature_2m_max    0
temperature_2m_min    0
precipitation_sum     0
windspeed_10m_max     0
weathercode           0
date                  0
dtype: int64
17:05:24  INFO      Sample:
   temperature_2m_max  temperature_2m_min  precipitation_sum  \
0                55.6                42.2              0.000   
1                50.4                38.8              0.669   
2                53.8                42.7              0.004   
3                58.2                36.5              0.110   
4                63.0                46.8              0.075   

   windspeed_10m_max  weathercode       date  
0               13.2            3 2024-01-01  
1               12.1           63 2024-01-02  
2               10.1           51 2024-01-03  
3               13.6           53 2024-01-04  
4               11.1           55 2024-01-05  


In [52]:
# Left-join daily weather 
df_baker_w = df_baker.merge(weather, on="date", how="left")

weather_nulls = df_baker_w["temperature_2m_max"].isnull().sum()
log.info("Baker + weather shape              : %s", df_baker_w.shape)
log.info("Calls with missing weather data   : %d", weather_nulls)

17:05:39  INFO      Baker + weather shape              : (82290, 33)
17:05:39  INFO      Calls with missing weather data   : 64891


In [53]:
log.info("df_baker_w shape : %s", df_baker_w.shape)
log.info("Null counts:\n%s", df_baker_w.isnull().sum())

17:05:45  INFO      df_baker_w shape : (82290, 33)
17:05:45  INFO      Null counts:
incident_number                     0
incident_type                       0
council_district                    0
mental_health_flag                  0
priority_level                      0
response_datetime                   0
response_year                       0
response_month                      0
response_day_of_week                0
response_hour                       0
first_unit_arrived_datetime         0
call_closed_datetime                0
sector                              0
initial_problem_description         0
initial_problem_category            0
final_problem_description           0
final_problem_category              0
number_of_units_arrived             0
unit_time_on_scene                  1
call_disposition_description        0
report_written_flag                 0
response_time                     645
officer_injured_killed_count        0
subject_injured_killed_count        0
othe

In [54]:
# Filter to 2024 and 2025
df_mongo = df_baker_w[
    df_baker_w["response_datetime"].dt.year.isin([2024, 2025])
].copy()

log.info("Filtered to 2024/2025 shape: %s", df_mongo.shape)

# select relevant columns
keep_cols = [
    "incident_number", "incident_type", "mental_health_flag", "priority_level",
    "response_datetime", "response_hour", "initial_problem_category",
    "final_problem_category", "number_of_units_arrived", "unit_time_on_scene",
    "response_time", "officer_injured_killed_count", "subject_injured_killed_count",
    "geo_id", "census_block_group", "date",
    # weather columns 
    "temperature_2m_max", "temperature_2m_min", "precipitation_sum",
    "windspeed_10m_max", "weathercode",
]
df_mongo = df_mongo[keep_cols]

# Attatch calendar and event features
df_mongo = add_features(df_mongo)

# Tag 2025 as train, 2024 as test
df_mongo["split"] = df_mongo["response_datetime"].dt.year.map({2025: "train", 2024: "test"})

log.info("Final df_mongo shape : %s", df_mongo.shape)
log.info("Columns              : %s", list(df_mongo.columns))
log.info("Split breakdown:\n%s", df_mongo["split"].value_counts())
log.info("Null counts:\n%s", df_mongo.isnull().sum())

17:07:09  INFO      Filtered to 2024/2025 shape: (17399, 33)
17:07:09  INFO      Final df_mongo shape : (17399, 34)
17:07:09  INFO      Columns              : ['incident_number', 'incident_type', 'mental_health_flag', 'priority_level', 'response_datetime', 'response_hour', 'initial_problem_category', 'final_problem_category', 'number_of_units_arrived', 'unit_time_on_scene', 'response_time', 'officer_injured_killed_count', 'subject_injured_killed_count', 'geo_id', 'census_block_group', 'date', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum', 'windspeed_10m_max', 'weathercode', 'day_of_week', 'month', 'day_of_year', 'is_weekend', 'season', 'is_holiday', 'is_ut_game', 'is_sxsw', 'is_acl', 'is_ut_movein', 'is_spring_break', 'is_graduation', 'split']
17:07:09  INFO      Split breakdown:
split
test     8946
train    8453
Name: count, dtype: int64
17:07:09  INFO      Null counts:
incident_number                   0
incident_type                     0
mental_health_flag        

In [56]:
# Connect to mongo atlas
try:
    client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
    client.admin.command("ping")  
    log.info("Connected to MongoDB: %s", DB_NAME)
except pymongo.errors.ConnectionFailure as e:
    log.error("MongoDB connection failed — check MONGO_URI in .env: %s", e)
    raise

db         = client[DB_NAME]
collection = db[COLLECTION_NAME]

# Prep docs
df_insert = df_mongo.copy()
# Serialize to ISO strings 
df_insert["response_datetime"] = df_insert["response_datetime"].astype(str)
df_insert["date"]              = df_insert["date"].astype(str)

# Use incident_number as MongoDB _id t
records = df_insert.to_dict(orient="records")
for record in records:
    record["_id"] = record.pop("incident_number")

# Insert
try:
    collection.drop()   # clear collection if exists
    result = collection.insert_many(records)
    log.info("Inserted %d documents into %s.%s",
             len(result.inserted_ids), DB_NAME, COLLECTION_NAME)
    log.info("Sample document:\n%s", collection.find_one())
except pymongo.errors.BulkWriteError as e:
    log.error("Bulk write error — %d write error(s)", len(e.details["writeErrors"]))
    raise
finally:
    client.close()
    log.info("MongoDB connection closed")

17:08:45  INFO      Connected to MongoDB: austin_911
17:08:48  INFO      Inserted 17399 documents into austin_911.calls
17:08:48  INFO      Sample document:
{'_id': 251541143, 'incident_type': 'Dispatched Incident', 'mental_health_flag': 'Not Mental Health Incident', 'priority_level': 'Priority 1', 'response_datetime': '2025-06-03 16:54:01', 'response_hour': 16, 'initial_problem_category': 'Other', 'final_problem_category': 'Other', 'number_of_units_arrived': 2.0, 'unit_time_on_scene': 3660.0, 'response_time': 429.0, 'officer_injured_killed_count': 0, 'subject_injured_killed_count': 0, 'geo_id': 484530323003.0, 'census_block_group': 4530323003.0, 'date': '2025-06-03', 'temperature_2m_max': 91.0, 'temperature_2m_min': 75.4, 'precipitation_sum': 0.075, 'windspeed_10m_max': 11.9, 'weathercode': 53.0, 'day_of_week': 1, 'month': 6, 'day_of_year': 154, 'is_weekend': 0, 'season': 'summer', 'is_holiday': 0, 'is_ut_game': 0, 'is_sxsw': 0, 'is_acl': 0, 'is_ut_movein': 0, 'is_spring_break': 0, 'i

## Mongosh verrifications in terminal

```js
// Connect and select the database
use austin_911

// Count total documents inserted
db.calls.countDocuments()


// Verify train / test split
db.calls.aggregate([{ $group: { _id: "$split", count: { $sum: 1 } } }])

// Priority distribution
db.calls.aggregate([{ $group: { _id: "$priority_level", count: { $sum: 1 } } }])

// Confirm weather columns are present and non-null
db.calls.findOne({}, {
    temperature_2m_max: 1, temperature_2m_min: 1,
    precipitation_sum: 1, windspeed_10m_max: 1, weathercode: 1
})

// Event flag spot-checks
db.calls.countDocuments({ is_sxsw: 1 })      
db.calls.countDocuments({ is_ut_game: 1 })      
db.calls.countDocuments({ is_acl: 1 })           
db.calls.countDocuments({ is_holiday: 1 })       

// Inspect one full doc
db.calls.findOne()
```